# Public coursework notebook using an open benchmark dataset
>
> Nothing in this notebook is taken from, derived from or capable of revealing confidential client or institutional transaction data. The dataset contains no real customer identifiers, card numbers or other personally identifiable information.
>
> The analysis uses the publicly available ULB Credit Card Fraud dataset (Kaggle, DbCL-1.0 licence). Most of its features (`V1`-`V28`) are anonymised PCA components supplied with the benchmark; only `Time` and `Amount` are in their original form.
>
> The notebook's purpose is to study *unsupervised anomaly detection* under severe class imbalance (fraud is very rare). It does not represent any specific bank's production fraud system.

---


# Detecting fraudulent card transactions with unsupervised anomaly detection

### CAM DS 2025 | Data Science with ML and AI | University of Cambridge PACE

---

## Project Overview

This notebook builds an unsupervised anomaly-detection pipeline for credit-card fraud using the ULB Credit Card Fraud benchmark.

The fraud labels are not used to train the models, reflecting real-world settings where labels are often incomplete, delayed or unreliable. Instead, the models learn what normal transactions look like and flag unusual ones for human review. The labels are used only afterwards, on held-out data, to measure how much of the known fraud those flags catch.

According to [UK Finance (2025)](https://www.ukfinance.org.uk/news-and-insight/press-release/fraud-report-2025-press-release), **criminals stole £1.17 billion through fraud in the UK in 2024**, broadly unchanged from 2023. Unauthorised fraud, where a criminal uses stolen card or account details, accounted for £722 million across 3.13 million cases. Authorised push payment (APP) scams, where the victim is tricked into sending the money themselves, accounted for £450.7 million. Worldwide, the [Nilson Report](https://nilsonreport.com/articles/card-fraud-losses-worldwide-2024/) puts payment card fraud losses alone at $33.41 billion in 2024, and projects about $408 billion in cumulative losses over the next ten years. Detecting fraud early therefore matters both financially and for an institution's reputation.

One cost imbalance shapes the modelling choices that follow: **a false negative (missed fraud) costs much more than a false positive (a legitimate transaction sent for manual review).**

The pipeline therefore favours recall, catching as much fraud as possible, and accepts more false alerts in return, as long as the number of alerts stays small enough for reviewers to handle.

Anomaly detection can also amplify bias in the data. Legitimate customers whose behaviour is uncommon, for example because their group is under-represented in the data, can look unusual to the model and be flagged more often than others.

Under GDPR and the UK Equality Act 2010, automated decisions that systematically disadvantage a group can be unlawful. Reviewers therefore need tools that explain why a transaction was flagged, and any personal data must stay anonymised and be used only for fraud prevention.

* **Data source:** ULB Credit Card Fraud dataset via Kaggle (`mlg-ulb/creditcardfraud`); public benchmark under DbCL-1.0
* **Domain:** card-transaction fraud detection; unsupervised anomaly detection under severe class imbalance
* **EDA focus:** missing values and duplicates, class imbalance, transaction amounts, fraud over time, correlation with the label (leakage check) and feature scales
* **Primary models and metric:** Isolation Forest, Local Outlier Factor (LOF) and One-Class SVM; evaluation prioritises recall and uses precision-recall analysis to choose the alert threshold

---



## Research Questions

This analysis is structured around five questions:

1. **What does "normal" look like in this dataset, and how well separated are the known fraud cases from normal transactions?**

   What do exploratory patterns in the PCA features, `Time` and `amount` (class imbalance, skew, correlation) suggest about how well an unsupervised detector can be expected to perform?

2. **Given that false negatives cost far more than false positives, how should models be evaluated and alert thresholds set?**

   How should the assumed fraud rate (the "contamination" setting), the ranking of anomaly scores and precision-recall analysis be used to prioritise recall without raising more alerts than reviewers can handle?

3. **Which unsupervised method best recovers fraud under that cost trade-off?**

   Do Isolation Forest, Local Outlier Factor and One-Class SVM differ materially in recall, precision and alert rate when evaluated in the same way?

4. **Are the supplied PCA components plus scaled `Time` and `amount` sufficient as model inputs, or does feature preparation change the results?**

   How should scaling and feature choices be recorded so that the models are compared fairly and no label information leaks into training?

5. **What robustness, fairness and governance constraints should limit automated flagging?**

   Where might rare but legitimate behaviour be over-flagged, what explanations do reviewers need, and how do GDPR and institutional ethics restrict deployment?

---

## Analytical Plan

The notebook follows a staged unsupervised fraud-detection pipeline.

Stages 1 to 3 (environment setup, data loading and exploratory analysis) are complete. Stages 4 to 12 are outlined below and will be implemented in later iterations.

**STAGE 1: Environment and package setup**<br>
Import libraries, set display options and a fixed random seed, and define the column-name constants used throughout the notebook.

**STAGE 2: Data loading and description**<br>
Download the ULB Credit Card Fraud dataset using the repository's Kaggle helper script, load it into a DataFrame, rename key columns consistently, and report the number of rows and columns and the fraud rate.

**STAGE 3: Exploratory data analysis**<br>
Check for missing values and duplicate rows, then examine class imbalance, transaction amounts, fraud over time, correlation with the label (and possible leakage) and feature scales: anything that may affect modelling.

**STAGE 4: Feature preparation**<br>
Build the model inputs: keep the PCA components, scale `Time` and `amount`, and document each choice so that no information from the label leaks into the training features.

**STAGE 5: Evaluation protocol**<br>
Set out how the labels are used for evaluation only: the assumed fraud rate (the "contamination" setting), how anomaly scores are ranked, and any train/test split that keeps training unsupervised.

**STAGE 6: Baseline anomaly detector**<br>
Fit a simple Isolation Forest as a baseline and record its recall (with supporting metrics) against the evaluation labels, as a reference point for later models.

**STAGE 7: Model comparison**<br>
Compare Isolation Forest, Local Outlier Factor and One-Class SVM using the same evaluation protocol, judging each on recall, precision and the number of alerts it raises.

**STAGE 8: Threshold selection**<br>
Use precision-recall analysis to choose an alert threshold that deliberately favours recall while keeping the number of false alarms manageable for reviewers.

**STAGE 9: Explainability**<br>
Explain each flagged case, for example by showing which features contributed most to its anomaly score, so reviewers can understand why an alert was raised.

**STAGE 10: Robustness, ethics and risks**<br>
Test how stable the results are, document bias and fairness risks, and record GDPR constraints on automated decisions and on the use of personal data.

**STAGE 11: Conclusions and recommendations**<br>
Summarise the strongest unsupervised approach, the trade-off between missed fraud and false alerts at the chosen threshold, and practical recommendations for the review process.

**STAGE 12: Reproducibility appendix**<br>
Record package versions and export a requirements snapshot so the analysis can be re-run end to end.



---

## 1. STAGE 1: Environment and package setup

Import libraries, set display options and a fixed random seed, and define the column-name constants used throughout the notebook.



### 1.1. Import required packages

Import the libraries used to load, check and plot the data, and fix the random seed so that results are reproducible.



In [ ]:
import gc
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("error")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=PendingDeprecationWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

SEED = 79
LABEL_COL = "label"
AMOUNT_COL = "amount"
TIME_COL = "Time"

np.random.seed(SEED)
plt.style.use("default")

print("Python", sys.version)
print("pandas", pd.__version__)


def release_memory():
    gc.collect()


---

## 2. STAGE 2: Data loading and description

Download the ULB Credit Card Fraud dataset using the repository's Kaggle helper script, load it into a DataFrame, rename key columns consistently, and report the number of rows and columns and the fraud rate.



### 2.1. Fetch and load the ULB extract

Download the public Kaggle dataset with `scripts/kaggle_fetch.py` (this needs your Kaggle API credentials saved as `.kaggle/kaggle.json` at the repository root). The script saves the file to `data/ulb/` at the repository root. Then load `creditcard.csv` and rename `Class` to `label` and `Amount` to `amount`, so the rest of the notebook uses consistent lower-case names.



In [ ]:
# Paths are relative to notebooks/, the folder this notebook runs from.
DATA_DIR = "../data/ulb"
!python ../scripts/kaggle_fetch.py mlg-ulb/creditcardfraud creditcard.csv {DATA_DIR}
df = pd.read_csv(f"{DATA_DIR}/creditcard.csv")
df.rename(columns={'Class':'label','Amount':'amount'}, inplace=True)
print("ULB shape:", df.shape); display(df.head())
print("Fraud rate:", df['label'].mean())

---

## 3. STAGE 3: Exploratory data analysis

Check for missing values and duplicate rows, then examine class imbalance, transaction amounts, fraud over time, correlation with the label (and possible leakage) and feature scales: anything that may affect modelling.

With the dataset's size and overall fraud rate confirmed, the checks below look at data quality and at how the features relate to the fraud label, before any modelling begins:

1. Missing values (3.1).
2. Duplicate rows (3.2).
3. Class imbalance (3.3).
4. Transaction amounts for legitimate and fraudulent transactions (3.4).
5. Fraud over time (3.6).
6. Correlation with the label, and the risk of leakage (3.8).
7. Feature scales (3.10).



### 3.1. Are any values missing?

A missing value is a blank cell: that piece of information was not recorded for the transaction. If an important column has many blanks, later analysis can be misleading. The check below counts the blanks in every column, including the fraud label.

In [ ]:
blank_counts = df.isna().sum()
blank_report = pd.DataFrame({
    "blanks": blank_counts,
    "pct_blank": df.isna().mean().mul(100),
}).sort_values("blanks", ascending=False, kind="stable")

display(blank_report.round({"pct_blank": 3}))
print(f"Total blank cells in the whole table: {blank_counts.sum():,}")

### 3.2. Are any rows repeated?

A duplicate is a row that appears more than once with identical values in every column. This can happen if the same transaction was recorded twice.

Here we only count duplicates; we do **not** remove any yet, and will decide how to handle them later.

In [ ]:
extra_copies = int(df.duplicated().sum())
rows_in_repeat_groups = int(df.duplicated(keep=False).sum())

print(f"Total rows: {len(df):,}")
print(f"Extra copies of an identical row: {extra_copies:,}")
print(f"Rows that belong to a repeated group (original + copies): {rows_in_repeat_groups:,}")
print("No rows were removed; this is a count only.")

### 3.3. How rare is fraud?

We expect fraud to be rare. When almost every transaction is legitimate, a model that always predicts "legitimate" achieves very high accuracy while catching no fraud at all, so accuracy alone is a misleading measure.

The counts below show how uneven the two classes are.



In [ ]:
class_counts = df[LABEL_COL].value_counts().sort_index()
class_counts.index = ["Legitimate (0)", "Fraud (1)"]
class_pct = class_counts.div(len(df)).mul(100)

summary = pd.DataFrame({"count": class_counts, "percent": class_pct.round(3)})
display(summary)

print(
    f"Out of {len(df):,} transactions, "
    f"{int(class_counts['Fraud (1)']):,} are fraud "
    f"({class_pct['Fraud (1)']:.3f}%)."
)

### 3.4. What do transaction amounts look like?

`amount` is the value of each transaction. Unlike the anonymised `V` columns, it can be understood without specialist knowledge. Comparing amounts for legitimate and fraudulent transactions shows whether fraud tends to be small, large or mixed.

In [ ]:
amount_by_class = (
    df.groupby(LABEL_COL)[AMOUNT_COL]
    .describe()
    .rename(index={0: "Legitimate", 1: "Fraud"})
)
display(amount_by_class.round(2))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

for ax, label_value, title, color in zip(
    axes,
    [0, 1],
    ["Legitimate amounts", "Fraud amounts"],
    ["steelblue", "darkorange"],
):
    values = df.loc[df[LABEL_COL] == label_value, AMOUNT_COL]
    ax.hist(values, bins=40, color=color, edgecolor="white")
    ax.set_title(title)
    ax.set_xlabel("Amount")
    ax.set_ylabel("Count")

plt.tight_layout()
plt.show()

leg_med = amount_by_class.loc["Legitimate", "50%"]
fraud_med = amount_by_class.loc["Fraud", "50%"]
leg_mean = amount_by_class.loc["Legitimate", "mean"]
fraud_mean = amount_by_class.loc["Fraud", "mean"]

print(
    f"Median amount: legitimate {leg_med:.2f}, fraud {fraud_med:.2f}. "
    f"Mean amount: legitimate {leg_mean:.2f}, fraud {fraud_mean:.2f}."
)

### 3.5. Analysis: Fraud amounts

Fraud amounts are **mixed**. The typical (median) fraudulent transaction is *smaller* than the typical legitimate one, but the *average* (mean) fraud is higher, because a small number of fraudulent transactions are large enough to pull the average up.

So there is no simple rule such as "fraud is always small" or "fraud is always large".

### 3.6. Does fraud look different over time?

`Time` is the number of seconds since the first transaction in the file, and the data covers about **two days**.

The real start time is unknown, so we assume the first transaction happened at 00:00 on Day 1. We then plot each day by **hour of day** (00:00-23:00) so the two days can be compared side by side.

*All clock times below are on this assumed clock, not real time of day.*

In [ ]:
eda_time = df[[TIME_COL, LABEL_COL]].copy()
elapsed_hour = (eda_time[TIME_COL] // 3600).astype(int)
eda_time["day"] = (elapsed_hour // 24) + 1
eda_time["hour_of_day"] = elapsed_hour % 24

by_day_hour = (
    eda_time.groupby(["day", "hour_of_day"])[LABEL_COL]
    .agg(n_txns="size", n_fraud="sum")
    .assign(fraud_rate_pct=lambda d: 100 * d["n_fraud"] / d["n_txns"])
)
volume = by_day_hour["n_txns"].unstack("day").rename(columns=lambda d: f"Day {d}")
fraud_rate = by_day_hour["fraud_rate_pct"].unstack("day").rename(columns=lambda d: f"Day {d}")

fig, axes = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True)
volume.plot(ax=axes[0], color=["steelblue", "darkorange"], lw=1.4)
axes[0].set_ylabel("All transactions")
axes[0].set_title("Transaction volume by hour of day")

fraud_rate.plot(ax=axes[1], color=["steelblue", "darkorange"], lw=1.4)
axes[1].set_ylabel("Fraud rate (%)")
axes[1].set_xlabel("Hour of day")
axes[1].set_title("Fraud rate by hour of day")
axes[1].set_xticks(range(0, 24, 2))
axes[1].set_xticklabels([f"{h:02d}:00" for h in range(0, 24, 2)])

plt.tight_layout()
plt.show()

### 3.7. Analysis: Fraud over time

The fraud rate **varies by hour**, and the pattern differs between the two days. Most hours sit close to the overall rate of about 0.17%, but a few are much higher (around 2% at 02:00 on Day 2) and some contain no fraud at all.

Because the fraud rate is not constant over time, the way we split the data for evaluation in Stage 5 (for example, by time rather than at random) may affect the results.

### 3.8. Which features move with the fraud label?

Correlation measures whether two columns tend to rise and fall together. Here we rank each feature by the strength of its correlation with the fraud label, ignoring whether the relationship is positive or negative.

This helps us understand the data, but it must not shape training: the label must never be a model input, and choosing features because they correlate with the label would also leak label information into training.

In [ ]:
feature_cols = [c for c in df.columns if c != LABEL_COL]
label_corr = (
    df[feature_cols]
    .corrwith(df[LABEL_COL])
    .abs()
    .sort_values(ascending=False)
)

top_corr = label_corr.head(10)
display(top_corr.to_frame("abs_corr_with_label").round(4))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(top_corr.index[::-1], top_corr.values[::-1], color="slategray")
ax.set_xlabel("Absolute correlation with fraud label")
ax.set_title("Top 10 features by correlation with the fraud label")
plt.tight_layout()
plt.show()


### 3.9. Analysis: Features and the fraud label

A few anonymised PCA features (notably `V17`, `V14` and `V12`) are much more strongly correlated with fraud than `Time` or `amount`. This suggests that fraudulent transactions do differ measurably from normal ones, which is encouraging for detection.

The models will nevertheless be trained without the label; these correlations are for understanding only.

### 3.10. Are `Time` and `amount` on a different scale from the `V` features?

The anonymised `V1`-`V28` columns come from PCA and are already centred on zero. `Time` and `amount` are raw values, in seconds and currency units respectively.

Anomaly detectors that rely on distances, such as LOF and One-Class SVM, can be dominated by columns with large values, so we compare the scale of each column before preparing features in Stage 4.

In [ ]:
v_cols = [c for c in df.columns if c.startswith("V")]
scale_report = pd.DataFrame(
    {
        "mean": df[[TIME_COL, AMOUNT_COL] + v_cols].mean(),
        "std": df[[TIME_COL, AMOUNT_COL] + v_cols].std(),
        "min": df[[TIME_COL, AMOUNT_COL] + v_cols].min(),
        "max": df[[TIME_COL, AMOUNT_COL] + v_cols].max(),
    }
)

display(scale_report.loc[[TIME_COL, AMOUNT_COL]].round(2))
display(scale_report.loc[v_cols].describe().round(2))


### 3.11. Analysis: Feature scales

`Time` and `amount` cover a much wider numeric range than the `V` features. For models based on distance or density (LOF and One-Class SVM), these two columns should be scaled in Stage 4 so they do not swamp the signal in the `V` features.

---

## 4. STAGE 4: Feature preparation

Build the model inputs: keep the PCA components, scale `Time` and `amount`, and document each choice so that no information from the label leaks into the training features.



---

## 5. STAGE 5: Evaluation protocol

Set out how the labels are used for evaluation only: the assumed fraud rate (the "contamination" setting), how anomaly scores are ranked, and any train/test split that keeps training unsupervised.



---

## 6. STAGE 6: Baseline anomaly detector

Fit a simple Isolation Forest as a baseline and record its recall (with supporting metrics) against the evaluation labels, as a reference point for later models.



---

## 7. STAGE 7: Model comparison

Compare Isolation Forest, Local Outlier Factor and One-Class SVM using the same evaluation protocol, judging each on recall, precision and the number of alerts it raises.



---

## 8. STAGE 8: Threshold selection

Use precision-recall analysis to choose an alert threshold that deliberately favours recall while keeping the number of false alarms manageable for reviewers.



---

## 9. STAGE 9: Explainability

Explain each flagged case, for example by showing which features contributed most to its anomaly score, so reviewers can understand why an alert was raised.



---

## 10. STAGE 10: Robustness, ethics and risks

Test how stable the results are, document bias and fairness risks, and record GDPR constraints on automated decisions and on the use of personal data.



---

## 11. STAGE 11: Conclusions and recommendations

Summarise the strongest unsupervised approach, the trade-off between missed fraud and false alerts at the chosen threshold, and practical recommendations for the review process.



---

## 12. STAGE 12: Reproducibility appendix

Record package versions and export a requirements snapshot so the analysis can be re-run end to end.

